In [1]:
import sys
sys.path.append("..")

from datetime import datetime
import numpy as np
from tqdm import tqdm
from torch.nn import TripletMarginLoss
import matplotlib.pyplot as plt
from lib.utils import load_transform
from copy import deepcopy
# from lib.features import WPTFeatureExtractor
from classes.feature import WPTFeatureExtractor
from typing import Tuple
import pickle 
import torch
from models.metric_learning import (
    DataLoader,
    MLPFeatureReducer,
    get_proba_per_sample,
    create_triplets_idx,
    TripletDataset,
    FeatureReducerScikit,
    MLPFeatureReducer
)

from sklearn.pipeline import Pipeline

In [2]:
plt.style.use("default")

In [3]:
PARAMS = {
    "level": 4,
    "family": "db16",
    "d": 2,
}

In [4]:
level = PARAMS["level"]
family = PARAMS["family"]
d = PARAMS["d"]

In [5]:
PATH_TO_VALID_DATASET = r"../raw/sequence_1/"
PATH_TO_TRAIN_DATASET = r"../raw/sequence_2/"
PATH_TO_TEST_DATASET = r"../raw/sequence_3/"

PATH_MODELS = r"../models/metric_learning/models_repo"

In [6]:
POSSIBLE_CABLES = [1,2,3,4]

In [11]:
transformer = WPTFeatureExtractor(
    level = level,
    family = family,
    reduction = "energy",
    normalize = True,
)

In [12]:
features_per_cable_train = load_transform(PATH_TO_TRAIN_DATASET, transformer)

100%|██████████| 4404/4404 [52:19<00:00,  1.40it/s]    


In [9]:
for cable in range(1,5):
    print(cable)
    cols = [c for c in features_per_cable_train[f"c{cable}"].columns if isinstance(c, int)]
    x_wpt = features_per_cable_train[f"c{cable}"][cols].to_numpy()
    reducer = MLPFeatureReducer(
        N=x_wpt.shape[1],
        d=2,
        hidden=32
    )
    reducer_scikit = FeatureReducerScikit(
        reducer_model=reducer,
        loss_fn = TripletMarginLoss(margin=1, p=2)
    )
    reducer_scikit.fit(x_wpt, num_epochs=50)

    output = {
        "feature_extraction": transformer,
        "feature_reduction": reducer,
        "params": PARAMS
    }
    filename = f"c{cable}_l{level}_f{family}_d{d}.pkl"

    with open(f"{PATH_MODELS}/{filename}", "wb") as h:
        pickle.dump(output, h, protocol=pickle.HIGHEST_PROTOCOL)
    
    del output

1


100%|██████████| 50/50 [00:37<00:00,  1.35it/s]


2


100%|██████████| 50/50 [00:36<00:00,  1.36it/s]


3


100%|██████████| 50/50 [00:36<00:00,  1.37it/s]


4


100%|██████████| 50/50 [00:36<00:00,  1.36it/s]
